# Diagnostic accuracy metrics: sensitivity, specificity, PPV, ROC and calibration

Exercise notebook for the lesson **[Diagnostic accuracy metrics: sensitivity, specificity, PPV, ROC and calibration](https://ramkumarmd.com/medical-ai/foundations/diagnostic-accuracy-metrics)** in the *Foundations* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Sensitivity and specificity describe how a test or model behaves in people with and without a condition. Predictive values tell you what a result means for the person in front of you, and they change with how common the condition is. AUROC measures how well a model ranks patients, and calibration checks whether its predicted probabilities can be taken at face value. A clinical AI model needs all of these, not one headline number.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Why PPV depends on prevalence

PPV for a test with 90% sensitivity and 80% specificity at different prevalences.

In [ ]:
def ppv(sensitivity, specificity, prevalence):
    true_pos = sensitivity * prevalence
    false_pos = (1 - specificity) * (1 - prevalence)
    return true_pos / (true_pos + false_pos)

for prevalence in [0.01, 0.05, 0.10, 0.30]:
    print(f"prevalence {prevalence:>4.0%}  PPV {ppv(0.90, 0.80, prevalence):.1%}")

# prevalence   1%  PPV 4.3%
# prevalence   5%  PPV 19.1%
# prevalence  10%  PPV 33.3%
# prevalence  30%  PPV 65.9%

### ROC curves and AUROC

Simulate a model's scores for 2,000 patients at 10% prevalence, then compute the metrics at a 0.5 threshold.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score

rng = np.random.default_rng(42)
n = 2000
y_true = rng.binomial(1, 0.10, size=n)  # 10% prevalence

# Patients with the condition get higher scores on average
logit = rng.normal(loc=np.where(y_true == 1, 1.5, -1.0), scale=1.0)
y_prob = 1 / (1 + np.exp(-logit))

y_pred = (y_prob >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f"Sensitivity {tp / (tp + fn):.2f}   Specificity {tn / (tn + fp):.2f}")
print(f"PPV         {tp / (tp + fp):.2f}   NPV         {tn / (tn + fn):.2f}")
print(f"AUROC       {roc_auc_score(y_true, y_prob):.3f}")

# Sensitivity 0.92   Specificity 0.85
# PPV         0.41   NPV         0.99
# AUROC       0.958

### Calibration: can the probabilities be trusted?

Continuing from the previous block: compare predicted and observed risk in ten groups, and compute the Brier score.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

observed, predicted = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
for p, o in zip(predicted, observed):
    print(f"predicted {p:.2f}   observed {o:.2f}")

print(f"Brier score {brier_score_loss(y_true, y_prob):.3f}")  # lower is better

# predicted 0.07   observed 0.00
# predicted 0.13   observed 0.00
# ...
# predicted 0.49   observed 0.04
# predicted 0.62   observed 0.20
# predicted 0.85   observed 0.76
# Brier score 0.120

## Exercise

**Goal:** Explore how threshold and prevalence change the metrics of a simulated risk model, and check its calibration.

**Data:** Synthetic patients generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Run the simulation above and record sensitivity, specificity, PPV and NPV at thresholds 0.3, 0.5 and 0.7.

In [ ]:
# Your code here


### Step 2

Re-run the simulation at 2% and 30% prevalence without changing the model. Which metrics move, and which stay roughly the same?

In [ ]:
# Your code here


### Step 3

Plot the ROC curve with sklearn.metrics.RocCurveDisplay and mark your three thresholds on it.

In [ ]:
# Your code here


### Step 4

Draw a calibration plot. Is the model over- or under-estimating risk? Recalibrate it with logistic (Platt) scaling and compare Brier scores.

In [ ]:
# Your code here


### Step 5

Write two sentences you would put in a report for clinicians describing this model's performance.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/foundations/diagnostic-accuracy-metrics](https://ramkumarmd.com/medical-ai/foundations/diagnostic-accuracy-metrics). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.